# 02 — Pré-processamento de Dados
### Tech Challenge — Fase 2 | Pós Tech Data Analytics (14DTAT) | Luis Guilherme Betarelli Lopes (RM374939)

Este notebook trata nulos, remove colunas sem valor preditivo, cria novas
features, separa treino/teste de forma estratificada e padroniza as
variáveis. Os artefatos gerados (`data/processed/*.csv`) alimentam os
notebooks `03_modelagem.ipynb` e `04_avaliacao.ipynb`.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

RANDOM_STATE = 42
os.makedirs("../data/processed", exist_ok=True)

In [2]:
df = pd.read_csv("../data/raw/WineQT.csv")
df.shape

(1143, 13)

## 1. Tratamento de dados faltantes e duplicados

In [3]:
print("Valores nulos por coluna:")
print(df.isnull().sum())
print("\nTotal de nulos na base:", df.isnull().sum().sum())
print("Linhas duplicadas:", df.duplicated().sum())

Valores nulos por coluna:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
Id                      0
dtype: int64

Total de nulos na base: 0
Linhas duplicadas: 0


**Documentação da decisão:** a base não tem nenhum valor nulo, então nenhuma
imputação é necessária. As linhas duplicadas foram mantidas — no contexto de
amostras de vinho, duas amostras com a mesma composição físico-química não são
necessariamente um erro de coleta, e removê-las artificialmente reduziria uma
base já pequena, sem ganho claro de qualidade dos dados.

## 2. Remoção de colunas sem valor preditivo e definição do alvo

In [4]:
df = df.drop(columns=["Id"])
df["quality_label"] = (df["quality"] >= 7).astype(int)
df[["quality", "quality_label"]].head()

,quality,quality_label
0,5,0
1,5,0
2,5,0
3,6,0
4,5,0


A coluna `Id` é apenas um identificador sequencial e foi removida. A
variável alvo `quality_label` já foi justificada no notebook 01 (limiar de 7,
definido pelo enunciado do desafio).

## 3. Feature engineering

Criamos duas novas features a partir de razões entre variáveis quimicamente
relacionadas, testando se capturam sinal adicional sobre a qualidade:

- `sulfur_dioxide_ratio`: proporção de dióxido de enxofre livre em relação ao
  total (indica o quanto do SO₂ ainda está disponível para proteger o vinho
  da oxidação).
- `acid_ratio`: razão entre acidez fixa e acidez volátil (relação entre a
  acidez "estrutural" desejável e a acidez associada a defeitos).

**Justificativa:** ambas combinam pares de variáveis que já se mostraram
relevantes na EDA (notebook 01) — a ideia é ver se a *relação* entre elas
carrega mais informação do que os valores brutos isoladamente.

In [5]:
df["sulfur_dioxide_ratio"] = df["free sulfur dioxide"] / df["total sulfur dioxide"]
df["acid_ratio"] = df["fixed acidity"] / df["volatile acidity"]

df[["sulfur_dioxide_ratio", "acid_ratio", "quality"]].corr()["quality"]

sulfur_dioxide_ratio    0.186621
acid_ratio              0.355357
quality                 1.000000
Name: quality, dtype: float64

As duas features criadas têm correlação relevante com a qualidade —
`acid_ratio` (0.36) inclusive supera a maioria das variáveis originais. Vamos
mantê-las no conjunto final de features e deixar que os próprios modelos (no
notebook 04) validem sua utilidade real via importância de variáveis.

## 4. Separação treino/teste

In [6]:
feature_cols = [c for c in df.columns if c not in ["quality", "quality_label"]]

X = df[feature_cols]
y = df["quality_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Proporção da classe 1 no treino:", round(y_train.mean(), 3))
print("Proporção da classe 1 no teste:  ", round(y_test.mean(), 3))

Treino: (800, 13) | Teste: (343, 13)
Proporção da classe 1 no treino: 0.139
Proporção da classe 1 no teste:   0.14


**Documentação — sem vazamento de dados:** usamos `stratify=y` para
preservar a proporção de vinhos de alta qualidade em treino e teste (base
desbalanceada, ~14%/86%, identificada no notebook 01). `RANDOM_STATE = 42` é
fixo para garantir reprodutibilidade. Nenhuma estatística é calculada no
conjunto de teste antes desta separação — o próximo passo (padronização) só
"aprende" a partir do treino.

## 5. Padronização das variáveis (Feature Scaling)

In [7]:
scaler = StandardScaler()
scaler.fit(X_train)  # média e desvio padrão calculados SOMENTE no treino

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train_scaled.describe().T[["mean", "std"]]

,mean,std
fixed acidity,-1.820766e-16,1.000626
volatile acidity,-2.375877e-16,1.000626
citric acid,-2.664535e-17,1.000626
residual sugar,-4.996004e-17,1.000626
chlorides,1.110223e-16,1.000626
free sulfur dioxide,7.771561e-17,1.000626
total sulfur dioxide,-4.218847e-17,1.000626
density,3.187894e-14,1.000626
pH,3.737011e-15,1.000626
sulphates,-1.287859e-16,1.000626


**Justificativa da normalização:** aplicamos padronização (z-score) com
`StandardScaler`, ajustada apenas no treino (para não vazar informação do
teste) e depois aplicada em treino e teste. É especialmente importante para o
KNN, que será treinado no próximo notebook e é sensível à escala das
variáveis por se basear em distância.

## 6. Exportação dos dados processados

In [8]:
X_train_scaled.assign(quality_label=y_train.values).to_csv("../data/processed/train.csv", index=False)
X_test_scaled.assign(quality_label=y_test.values).to_csv("../data/processed/test.csv", index=False)
joblib.dump(scaler, "../data/processed/scaler.pkl")

print("Arquivos salvos em data/processed/: train.csv, test.csv, scaler.pkl")

Arquivos salvos em data/processed/: train.csv, test.csv, scaler.pkl


## Resumo para o próximo notebook

- `data/processed/train.csv` e `data/processed/test.csv` contêm as features já
  padronizadas e a coluna `quality_label` (alvo).
- `data/processed/scaler.pkl` guarda o `StandardScaler` ajustado, caso seja
  necessário transformar novos dados.
- Próximo passo (`03_modelagem.ipynb`): treinar e comparar pelo menos dois
  modelos de classificação usando `train.csv`.